# MP3 AI Türkçe Dublaj — Colab
**Runtime > Change runtime type > GPU** seçin.

Akış: repo'yu çek → bağımlılıklar → girdi (YouTube linki **veya** dosya) → dublaj → sonucu incele/indir → (opsiyonel) sonucu GitHub'a geri gönder.

## 1. Repo + sistem bağımlılıkları

In [ ]:
!apt-get -qq install -y ffmpeg
!pip install -q yt-dlp
!git clone https://github.com/0xR0/dublaj.git || true
%cd dublaj

## 2. Python bağımlılıkları (birkaç dakika sürer)

In [ ]:
!pip install -q -r requirements.txt

## 3. Ortam değişkenleri
- **HF_TOKEN** (opsiyonel): pyannote ile daha iyi konuşmacı ayırma. Ücretsiz almak için: https://huggingface.co/settings/tokens → 'New token' (read yetkisi) → ayrıca https://huggingface.co/pyannote/speaker-diarization-3.1 ve https://huggingface.co/pyannote/segmentation-3.0 sayfalarındaki şartları 'Agree' ile kabul et. Boş bırakırsan **speechbrain** kullanılır (token gerekmez).

In [ ]:
import os
os.environ['HF_TOKEN'] = ''            # ucretsiz HF token (opsiyonel)
os.environ['COQUI_TOS_AGREED'] = '1'   # XTTS lisans kabulu

## 4-A. Girdi: YouTube linkinden ses indir
Linki yapıştır; yt-dlp sesi indirip MP3'e çevirir.

In [ ]:
YT_URL = "https://www.youtube.com/watch?v=XXXXXXXXXXX"  # <-- linki buraya
!yt-dlp -x --audio-format mp3 --audio-quality 0 -o "input/yt_audio.%(ext)s" "$YT_URL"
INPUT = "input/yt_audio.mp3"
print("Girdi:", INPUT)

## 4-B. (Alternatif) Girdi: kendi dosyanı yükle
YouTube yerine elindeki dosyayı kullanacaksan bu hücreyi çalıştır.

In [ ]:
from google.colab import files
import shutil
up = files.upload()
name = list(up.keys())[0]
shutil.move(name, f'input/{name}')
INPUT = f'input/{name}'
print('Girdi:', INPUT)

## 5. Dublajı çalıştır
İlk çalıştırmada modeller indirilir (XTTS ~1.8GB). Uzun sürebilir.
Bayraklar: `--no-background` (hızlı), `--tts edge` (XTTS yerine yedek), `--diarizer speechbrain|pyannote`.

In [ ]:
!python dub.py "$INPUT"

## 6. Sonucu incele

In [ ]:
!ls -la output/
print('--- run.log (son 40 satir) ---')
!tail -n 40 logs/run.log

## 7. Sonucu indir

In [ ]:
from google.colab import files
files.download('output/output_dubbed.mp3')

## 8. (Opsiyonel) Sonucu GitHub'a geri gönder
Böylece Termux tarafında `git pull` ile commitler + log + çıktı incelenebilir.
GitHub token: https://github.com/settings/tokens → fine-grained → 0xR0/dublaj deposuna **Contents: Read and write** yetkisi.

In [ ]:
GITHUB_TOKEN = ""  # <-- write yetkili token
!git config user.email "colab@dublaj"
!git config user.name "colab"
!git add -f output/output_dubbed.mp3 logs/run.log
!git commit -m "colab: dublaj sonucu + loglar" || echo "degisiklik yok"
!git push https://$GITHUB_TOKEN@github.com/0xR0/dublaj.git HEAD:colab-results